### ライブラリの準備

###モジュールのインポートとGoogleドライブのマウント

In [ ]:
import os
import torch.autograd
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import datetime
#from tqdm import tqdm
#from tqdm.notebook import tqdm
from tqdm.notebook import tqdm
import pickle
import copy
import gc
import random
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from PIL import Image
#import skimage.transform
from collections import deque
from typing import Sequence, Dict, Tuple, Union, Optional
#from argparse import Namespace
from dataclasses import dataclass, field
#from sacrebleu.metrics import BLEU
from evaluate import load
#import jiwer
#from comet import download_model, load_from_checkpoint

import torch
import kornia.augmentation as K
from kornia.augmentation.auto import AutoAugment
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torchvision import models
import torchvision.transforms as T
import torchvision.datasets as dataset
from torchvision.transforms import v2

#from transformers import  get_linear_schedule_with_warmup
from transformers import get_cosine_schedule_with_warmup

#from transformers import AutoImageProcessor, AutoModel, AutoProcessor, CLIPVisionModel
#from transformers import AutoTokenizer, CLIPVisionModel, AutoModelForCausalLM
from transformers import BertTokenizer, BertModel, CLIPVisionModel, BertForPreTraining

import sys
from evaluate import load

import util
#import levenshtein
#from nltk import bleu_score
from torchmetrics.multimodal import CLIPScore
import ssl
from torch.amp import autocast, GradScaler
from collections import OrderedDict
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider
#import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from pycocoevalcap.cider.cider import Cider
import json
import collections
from collections import Counter
import plotly
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
#fn = bleu_score.SmoothingFunction().method7
import time
#from pycocoevalcap.spice.spice import Spice
#import bitsandbytes as bnb
#logging.getLogger('rouge_score.rouge_scorer').setLevel(logging.WARNING)
#logging.set_verbosity_warning()

In [ ]:
class PositionalEmbedding(nn.Module):
    '''
    位置埋め込み （Positional embedding）
    dim_embedding: 埋込み次元
    max_len      : 入力の最大系列長
    '''
    def __init__(self, dim_embedding: int, max_len: int=2048):
        super().__init__()

        self.pos_emb = nn.Embedding(max_len, dim_embedding)

    '''
    位置エンコーディングの順伝播
    x: 位置エンコーディングを埋め込む対象のテンソル,
       [バッチサイズ, 系列長, 埋め込み次元]
    '''
    def forward(self, x: torch.Tensor):
        seq = x.shape[1]
        positions = torch.arange(start=0, end=seq, step=1, device=x.device).to(torch.long)
        positions = self.pos_emb(positions)[:seq,:]
        
        return positions

In [ ]:
model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
sos_token_id = tokenizer.encode( [ "[unused0]" ] )[1]
eos_token_id = tokenizer.encode( [ "[unused1]" ] )[1]
#sos_token_id = 1
#eos_token_id = 2
a_token_id = tokenizer.encode( "a"  )[1]
#print( a_token_id )
the_token_id = tokenizer.encode( "the" )[1]
and_token_id = tokenizer.encode( "and" )[1]
in_token_id = tokenizer.encode( "in" )[1]
we_token_id = tokenizer.encode( "we" )[1]
i_token_id = tokenizer.encode( "i" )[1]
he_token_id = tokenizer.encode( "he" )[1]
she_token_id = tokenizer.encode( "she" )[1]
it_token_id = tokenizer.encode( "it" )[1]
they_token_id = tokenizer.encode( "they" )[1]
period_token_id = tokenizer.encode( "." )[1]
comma_token_id = tokenizer.encode( "," )[1]
dbl_token_id = tokenizer.encode( '"' )[1]
sgl_token_id = tokenizer.encode( "'" )[1]

# 辞書サイズを保存
vocab_size = len( tokenizer )

class ComputeReward(nn.Module):
    def __init__(self, reward_t = 'ordinary', decode_t = 'ordinary', device="cpu", 
                 repeat_thresh = (3,2,2,2,2), repeat_weight = (1, 1, 1, 1, 1), cider_coef = 1.0, rouge_coef = 1.0, clip_coef = 1.0, 
                 bert_coef = 1.0, use_amp = True ):
        super().__init__()
        self.tokenizer = tokenizer
        self.tgt_lang = "en"
        self.device = device

        self.scorer = Cider()
        self.rougeL = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
        model_name = "distilbert-base-uncased"
        self.bert = load('bertscore' )
        self.metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(self.device)
        for param in self.metric.parameters():
            param.requires_grad = False
        self.reward_t = reward_t
        self.repeat_thresh = repeat_thresh
        self.repeat_weight = repeat_weight
        self.decode_t = decode_t
        self.cider_coef = cider_coef
        self.rouge_coef = rouge_coef
        self.clip_coef = clip_coef
        self.bert_coef = bert_coef
        self.use_amp = use_amp
    
    def _compute_reward_ord(self, preds, targets, imgs2, sources=None):
        """
        Compute reward metric for a batch of prediction and target sentences
        """
        model_name = "distilbert-base-uncased"
        # detokenize (convert to str) preds & targets
        if self.decode_t == 'no-endoftext':
            preds_str = [self.tokenizer.decode(
                [pred[i] for i in range( 1,  len( pred )  ) if not (pred[i-1] == endoftext_token_id and pred[i] == endoftext_token_id) ]
                ) for pred in preds]
            targets_str = [self.tokenizer.decode(
                [target[i] for i in range( 1,  len( target )  ) if not (target[i-1] == endoftext_token_id and target[i] == endoftext_token_id) ]
                ) for target in targets]
        elif self.decode_t == 'no-pad':
            preds1 = preds.clone().to(self.device )
            preds3 = preds.clone().to(self.device )
            targets1 = targets.clone().to(self.device )
            #preds1[preds1 == eos_token_id] = pad_token_id
            decoded = self.tokenizer.batch_decode(preds1, skip_special_tokens=False)
            #preds_str = [s.replace("[PAD]", "").replace("[unused1]", "").strip() for s in decoded]  
            preds_str = [s.replace("[PAD]", "").strip() for s in decoded]
            preds3[preds3 == eos_token_id] = pad_token_id
            preds_str2 = self.tokenizer.batch_decode(preds3, skip_special_tokens = True )
            #targets1[targets1 == eos_token_id] = pad_token_id
            decoded = self.tokenizer.batch_decode(targets1, skip_special_tokens=False)
            #targets_str = [s.replace("[PAD]", "").replace("[unused1]", "").strip() for s in decoded]            
            targets_str = [s.replace("[PAD]", "").strip() for s in decoded]            
        else:
            preds_str = [self.tokenizer.decode(pred) for pred in preds]
            targets_str = [self.tokenizer.decode(target) for target in targets]
        sources_str = [self.tokenizer.decode(source, ref="src") for source in sources] if sources is not None else None
        #print( "preds size:", preds.size() )
        #print( "targets size:", targets.size() )
        
        #print(f'1st target sent: {targets_str[0]}')
        #print(f'1st pred sent: {preds_str[0]}')
        #print(f'1st pred2 sent: {preds_str2[0]}' )
        
        # compute reward metric
        seq_len = preds.shape[1]

        pred_dict = { str(i): [item] for i, item in enumerate( preds_str)}
        target_dict = { str(i): [item] for i, item in enumerate( targets_str)}
        score, scores = self.scorer.compute_score(target_dict, pred_dict)
        #reward_cider = torch.tensor( scores ).to( self.device )[:,None].expand( -1, seq_len )
        reward_cider = torch.tensor( scores ).to( self.device )[:,None]
       
        #reward_rouge = [[self.rougeL.score(target, pred)['rougeL'][0]]  * seq_len for pred, target in zip(preds_str, targets_str)]
        reward_rouge = [[self.rougeL.score(target, pred)['rougeL'][0]] for pred, target in zip(preds_str, targets_str)]
        reward_rouge = torch.tensor( reward_rouge ).to( self.device )
        with autocast(str(self.device),enabled=self.use_amp):
            with torch.no_grad():
                processed = self.metric.processor(text=preds_str2, images=imgs2, return_tensors="pt", padding=True, \
                                                  truncation=True, max_length=77, do_resize=False, do_rescale=False ).to(self.device)
                outputs = self.metric.model(**processed)
                # 特徴量の正規化
                image_features = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
                text_features = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
                individual_scores = torch.clamp( (image_features.to(self.device) * \
                                                  text_features.to(self.device)).sum(axis=-1), min=0)
                #clip_scores = individual_scores[:,None].expand( -1, seq_len ) / 100.0
                clip_scores = individual_scores[:,None] / 100.0
                reward_clip = clip_scores
                model_name = 'distilbert-base-uncased' 
                bert_scores = self.bert.compute(
                    predictions=preds_str, 
                    references=targets_str,
                    model_type=model_name,
                    use_fast_tokenizer=True, 
                    lang='en', 
                    device=self.device,
                    batch_size=config.batch_size * config.num_samples,  # メモリ許容範囲で大きく設定
                    rescale_with_baseline=False
                )['f1']
                #reward_bert = torch.tensor( bert_scores ).to( self.device )[:,None].expand( -1, seq_len )
                reward_bert = torch.tensor( bert_scores ).to( self.device )[:,None]
        reward = self.cider_coef * reward_cider + self.rouge_coef * reward_rouge \
            + self.clip_coef * reward_clip + self.bert_coef * reward_bert
        reward2 = reward_cider + reward_rouge + reward_bert + reward_clip
        
        return reward, reward2
    '''
    def compute_length_reward(self, preds, targets):
        # preds.size(1) を L として取得
        max_len = preds.size(1)
        # 1から始まるインデックスを作成 (長さとして計算するため)
        arange_index = torch.arange(1, max_len + 1, device=self.device).float()
        
        def get_length(tokens):
            # eosの位置を特定
            eos_mask = (tokens == eos_token_id)
            # 最初のeosだけを抽出
            first_eos = (eos_mask.int().cumsum(dim=1) == 1) & eos_mask
            
            # eosがある場合はその位置(1-indexed)を、ない場合は最大長を返す
            lengths = torch.sum(first_eos.float() * arange_index, dim=1)
            # eosが一度も出現しなかった行は 0 になっているので、max_len で埋める
            no_eos_mask = (eos_mask.sum(dim=1) == 0)
            lengths[no_eos_mask] = float(max_len)
            
            # 正規化 (0.0 ~ 1.0)
            return lengths / max_len

        pred_lengths = get_length(preds)
        target_lengths = get_length(targets)
        #
        # MSEの計算 (負の値を報酬とする)
        # reduction='none' なので [batch_size] の形
        #reward_lengths = - ((pred_lengths - target_lengths) ** 2)
        reward_lengths = - torch.abs( pred_lengths - target_lengths) * 5
        reward_lengths = reward_lengths - ( pred_lengths > 0.4 ) * 10
        
        # [batch_size, 1] にしてから [batch_size, seq_len] に拡張
        #reward = reward_lengths.unsqueeze(1).expand(-1, max_len)
        reward = reward_lengths.unsqueeze(1)
        
        return reward

    def unique_ngram_ratio(self, preds):
        bsz, seq_len = preds.size()
        ng = 5
        device = preds.device
    
        # 1. 各シーケンスの有効な長さを特定 (eosまで)
        # cumsumを使って最初のeos以降をマスクする
        eos_mask = (preds == eos_token_id)
        first_eos_idx = (eos_mask.cumsum(dim=1) == 1) & eos_mask
        # eosがない場合はseq_lenとする
        lengths = torch.where(first_eos_idx.any(dim=1), 
                              first_eos_idx.float().argmax(dim=1), 
                              torch.tensor(seq_len, device=device)).unsqueeze(1)
    
        unr_list = []
        
        # n-gramのサイズごとに一括処理
        for n in range(1, ng + 1):
            # unfoldで全n-gramを抽出: (bsz, seq_len - n + 1, n)
            if seq_len < n:
                unr_list.append(torch.zeros(bsz, 1, device=device))
                continue
                
            ngrams = preds.unfold(1, n, 1)
            num_ngrams = ngrams.size(1)
    
            # マスク作成: 有効な長さ（eos前）に含まれるn-gramのみを残す
            # n-gramの開始位置が (length - n + 1) 未満である必要がある
            ngram_indices = torch.arange(num_ngrams, device=device).expand(bsz, -1)
            valid_mask = ngram_indices < (lengths - n + 1)
            
            # 非常に大きい値で無効なn-gramを埋める（uniqueカウントから除外するため）
            # または、バッチを跨いでユニーク判定するためにハッシュ化
            # ここでは各バッチ行ごとにユニーク数を数える必要があるため、
            # 完全なベクトル化には torch.unique の制限（バッチ非対応）を回避する工夫が必要
            
            # 解決策: 各行をユニークなオフセットでシフトして全体でuniqueをとる手法
            # ただし、シンプルさとメモリ効率のため、ngループのみ残すのが現実的です。
            
            # 行ごとのUniqueカウント (Vectorized version of unique per row)
            # 完全にforを消す場合、各n-gramを1つのスカラにパッキング(ハッシュ)して処理
            if n == 1:
                packed = ngrams.squeeze(-1)
            else:
                # 各要素を大きな基数でシフトして足し合わせ、1つの整数にする
                max_val = preds.max() + 1
                coeffs = max_val ** torch.arange(n, device=device)
                packed = (ngrams * coeffs).sum(dim=-1)
    
            # 無効な位置にユニークな（重ならない）負の値を代入
            invalid_fill = -1 - torch.arange(bsz * num_ngrams, device=device).reshape(bsz, num_ngrams)
            packed = torch.where(valid_mask, packed.float(), invalid_fill.float())
    
            # 行ごとにユニーク数をカウント
            # Note: PyTorchのuniqueはバッチ未対応なため、以下が最速の回避策の一つ
            def count_unique_rowwise(t, mask):
                # ソートして隣接要素との差を見ることでユニーク数を算出
                t_sorted, _ = torch.sort(t, dim=1)
                diffs = (t_sorted[:, 1:] != t_sorted[:, :-1]).int()
                # 最初の要素 + 変化があった回数 - 無効値の数(バッチ内の無効分を補正)
                unique_counts = diffs.sum(dim=1) + 1
                # 無効な埋め草（すべてユニークに設定済み）の数を引いて、
                # 有効なものが1つもなければ0にする処理
                invalid_count = (~mask).sum(dim=1)
                return (unique_counts - invalid_count).float()
    
            row_unique = count_unique_rowwise(packed, valid_mask)
            row_total = valid_mask.sum(dim=1).clamp(min=1).float()
            unr_list.append(row_unique / row_total)
    
        # 結果の整形
        unr_tensor = torch.stack(unr_list, dim=1) * 20 # (bsz, ng) 
        #return torch.mean(unr_tensor, dim=1)[:, None].expand(-1, seq_len)
        return torch.mean(unr_tensor, dim=1)[:, None]

    def calc_ngram_repeat_fast(self, preds):
        # preds が書き換わらないよう、この関数内では元の値を保護する
        bsz, seq_len = preds.size()
        ngram_cnt = torch.zeros(bsz, device=preds.device, dtype=torch.float)
        
        base_ignore = [pad_token_id, eos_token_id, cls_token_id, sep_token_id]
        extra_ignore = [a_token_id, the_token_id, period_token_id, comma_token_id, and_token_id, in_token_id]
    
        for n in range(1, 5):
            if seq_len < n:
                continue
    
            current_ignore_ids = base_ignore + (extra_ignore if n == 1 else [])
            # ignore_mask は bool なので preds に影響しません
            ignore_mask = torch.isin(preds, torch.tensor(current_ignore_ids, device=preds.device))
            
            # 【重要】unfoldの後に .clone() を入れてメモリを切り離す
            ngrams = preds.unfold(dimension=1, size=n, step=1).clone()
            num_ngrams = ngrams.size(1)
            
            ngram_ignore_mask = ignore_mask.unfold(dimension=1, size=n, step=1).any(dim=-1)
            
            if n > 1:
                # 語彙サイズに基づくハッシュ化
                vocab_size_max = max(preds.max().item(), 100000)
                weights = torch.pow(torch.tensor([vocab_size_max], device=preds.device), 
                                    torch.arange(n, device=preds.device)).long()
                hashed_ngrams = (ngrams.long() * weights).sum(dim=-1)
            else:
                hashed_ngrams = ngrams.squeeze(-1).long()
    
            # これで hashed_ngrams を書き換えても、clone 済みのデータなので preds は安全
            invalid_val = -1
            hashed_ngrams[ngram_ignore_mask] = invalid_val
    
            for b in range(bsz):
                b_ngrams = hashed_ngrams[b]
                valid_b_ngrams = b_ngrams[b_ngrams != invalid_val]
                
                if valid_b_ngrams.numel() == 0:
                    continue
                    
                unique_vals, counts = torch.unique(valid_b_ngrams, return_counts=True)
                
                mask = counts >= self.repeat_thresh[n-1]
                ngram_cnt[b] += counts[mask].sum().float() * self.repeat_weight[n-1]
    
        #normalized_cnt = ngram_cnt / 10.0  

        ## 2. Sigmoid関数で0〜1の範囲に圧縮し、それをペナルティ係数とする
        ##    これにより、ngram_cntが増えてもペナルティは最大20(負の方向)に収束する
        #penalty_factor = torch.sigmoid(normalized_cnt * 5 - 3) # ハイパーパラメータは調整可能
        #penalty = - penalty_factor * 20

        penalty = - (torch.tanh(ngram_cnt / 3.0) * 20)

        #penalty = - torch.pow(ngram_cnt, 2.0) / seq_len * 5.0

        #penalty = - torch.nn.functional.softplus(torch.pow(1.5, ngram_cnt) - 1.0) / seq_len * 10.0

        #penalty = - (torch.expm1(ngram_cnt / 3.0)) / seq_len * 20.0
        
        #penalty = - torch.clamp(torch.pow(2.0, ngram_cnt - 1.0) / seq_len, max=1.0) * 20
        #penalty = - torch.pow(2.0, ngram_cnt - 1.0) / seq_len * 20
        #return penalty[:, None].expand(-1, seq_len)
        return penalty[:, None]
    '''
    def compute_length_reward(self, preds, targets):
        # preds.size(1) を L として取得
        max_len = preds.size(1)
        # 1から始まるインデックスを作成 (長さとして計算するため)
        arange_index = torch.arange(1, max_len + 1, device=self.device).float()
        
        def get_length(tokens):
            # eosの位置を特定
            eos_mask = (tokens == eos_token_id)
            # 最初のeosだけを抽出
            first_eos = (eos_mask.int().cumsum(dim=1) == 1) & eos_mask
            
            # eosがある場合はその位置(1-indexed)を、ない場合は最大長を返す
            lengths = torch.sum(first_eos.float() * arange_index, dim=1)
            # eosが一度も出現しなかった行は 0 になっているので、max_len で埋める
            no_eos_mask = (eos_mask.sum(dim=1) == 0)
            lengths[no_eos_mask] = float(max_len)
            
            # 正規化 (0.0 ~ 1.0)
            return lengths / max_len

        pred_lengths = get_length(preds)
        target_lengths = get_length(targets)
        #
        # MSEの計算 (負の値を報酬とする)
        # reduction='none' なので [batch_size] の形
        reward_lengths = - ((pred_lengths - target_lengths) ** 2)
        
        # [batch_size, 1] にしてから [batch_size, seq_len] に拡張
        #reward = reward_lengths.unsqueeze(1).expand(-1, max_len)
        reward = reward_lengths.unsqueeze(1)
        
        return reward

    def unique_ngram_ratio(self, preds):
        bsz, seq_len = preds.size()
        ng = 4
        device = preds.device
    
        # 1. 各シーケンスの有効な長さを特定 (eosまで)
        # cumsumを使って最初のeos以降をマスクする
        eos_mask = (preds == eos_token_id)
        first_eos_idx = (eos_mask.cumsum(dim=1) == 1) & eos_mask
        # eosがない場合はseq_lenとする
        lengths = torch.where(first_eos_idx.any(dim=1), 
                              first_eos_idx.float().argmax(dim=1), 
                              torch.tensor(seq_len, device=device)).unsqueeze(1)
    
        unr_list = []
        
        # n-gramのサイズごとに一括処理
        for n in range(1, ng + 1):
            # unfoldで全n-gramを抽出: (bsz, seq_len - n + 1, n)
            if seq_len < n:
                unr_list.append(torch.zeros(bsz, 1, device=device))
                continue
                
            ngrams = preds.unfold(1, n, 1)
            num_ngrams = ngrams.size(1)
    
            # マスク作成: 有効な長さ（eos前）に含まれるn-gramのみを残す
            # n-gramの開始位置が (length - n + 1) 未満である必要がある
            ngram_indices = torch.arange(num_ngrams, device=device).expand(bsz, -1)
            valid_mask = ngram_indices < (lengths - n + 1)
            
            # 非常に大きい値で無効なn-gramを埋める（uniqueカウントから除外するため）
            # または、バッチを跨いでユニーク判定するためにハッシュ化
            # ここでは各バッチ行ごとにユニーク数を数える必要があるため、
            # 完全なベクトル化には torch.unique の制限（バッチ非対応）を回避する工夫が必要
            
            # 解決策: 各行をユニークなオフセットでシフトして全体でuniqueをとる手法
            # ただし、シンプルさとメモリ効率のため、ngループのみ残すのが現実的です。
            
            # 行ごとのUniqueカウント (Vectorized version of unique per row)
            # 完全にforを消す場合、各n-gramを1つのスカラにパッキング(ハッシュ)して処理
            if n == 1:
                packed = ngrams.squeeze(-1)
            else:
                # 各要素を大きな基数でシフトして足し合わせ、1つの整数にする
                max_val = preds.max() + 1
                coeffs = max_val ** torch.arange(n, device=device)
                packed = (ngrams * coeffs).sum(dim=-1)
    
            # 無効な位置にユニークな（重ならない）負の値を代入
            invalid_fill = -1 - torch.arange(bsz * num_ngrams, device=device).reshape(bsz, num_ngrams)
            packed = torch.where(valid_mask, packed.float(), invalid_fill.float())
    
            # 行ごとにユニーク数をカウント
            # Note: PyTorchのuniqueはバッチ未対応なため、以下が最速の回避策の一つ
            def count_unique_rowwise(t, mask):
                # ソートして隣接要素との差を見ることでユニーク数を算出
                t_sorted, _ = torch.sort(t, dim=1)
                diffs = (t_sorted[:, 1:] != t_sorted[:, :-1]).int()
                # 最初の要素 + 変化があった回数 - 無効値の数(バッチ内の無効分を補正)
                unique_counts = diffs.sum(dim=1) + 1
                # 無効な埋め草（すべてユニークに設定済み）の数を引いて、
                # 有効なものが1つもなければ0にする処理
                invalid_count = (~mask).sum(dim=1)
                return (unique_counts - invalid_count).float()
    
            row_unique = count_unique_rowwise(packed, valid_mask)
            row_total = valid_mask.sum(dim=1).clamp(min=1).float()
            unr_list.append(row_unique / row_total)
    
        # 結果の整形
        unr_tensor = torch.stack(unr_list, dim=1) # (bsz, ng)
        #return torch.mean(unr_tensor, dim=1)[:, None].expand(-1, seq_len)
        return torch.mean(unr_tensor, dim=1)[:, None]

    def calc_ngram_repeat_fast(self, preds):
        # preds が書き換わらないよう、この関数内では元の値を保護する
        bsz, seq_len = preds.size()
        ngram_cnt = torch.zeros(bsz, device=preds.device, dtype=torch.float)
        
        base_ignore = [pad_token_id, eos_token_id, cls_token_id, sep_token_id]
        extra_ignore = [a_token_id, the_token_id, period_token_id, comma_token_id, and_token_id, in_token_id]

        ng = 4        
        
        for n in range(1, ng + 1 ):
            if seq_len < n:
                continue
    
            current_ignore_ids = base_ignore + (extra_ignore if n == 1 else [])
            # ignore_mask は bool なので preds に影響しません
            ignore_mask = torch.isin(preds, torch.tensor(current_ignore_ids, device=preds.device))
            
            # 【重要】unfoldの後に .clone() を入れてメモリを切り離す
            ngrams = preds.unfold(dimension=1, size=n, step=1).clone()
            num_ngrams = ngrams.size(1)
            
            ngram_ignore_mask = ignore_mask.unfold(dimension=1, size=n, step=1).any(dim=-1)
            
            if n > 1:
                # 語彙サイズに基づくハッシュ化
                vocab_size_max = max(preds.max().item(), 100000)
                weights = torch.pow(torch.tensor([vocab_size_max], device=preds.device), 
                                    torch.arange(n, device=preds.device)).long()
                hashed_ngrams = (ngrams.long() * weights).sum(dim=-1)
            else:
                hashed_ngrams = ngrams.squeeze(-1).long()
    
            # これで hashed_ngrams を書き換えても、clone 済みのデータなので preds は安全
            invalid_val = -1
            hashed_ngrams[ngram_ignore_mask] = invalid_val
    
            for b in range(bsz):
                b_ngrams = hashed_ngrams[b]
                valid_b_ngrams = b_ngrams[b_ngrams != invalid_val]
                
                if valid_b_ngrams.numel() == 0:
                    continue
                    
                unique_vals, counts = torch.unique(valid_b_ngrams, return_counts=True)
                
                #mask = counts >= self.repeat_thresh[n-1]
                #ngram_cnt[b] += counts[mask].sum().float() * self.repeat_weight[n-1]
                ## しきい値以上のものを取り出す
                #mask = counts >= self.repeat_thresh[n-1]
                ## しきい値を超えた分（リピート分）だけを抽出して加算
                ## 例: thresh=2 で 3回出たら、3 - (2-1) = 2回分。
                ## もし「最初の1回は常に無料」にしたいなら、さらに -1 する等の調整もアリです。
                #repeat_counts = counts[mask] - (self.repeat_thresh[n-1] - 1)
                #ngram_cnt[b] += repeat_counts.sum().float() * self.repeat_weight[n-1]

                mask = counts > self.repeat_thresh[n-1] 
                if mask.any():
                    # 超過した数だけを純粋に加算する
                    # 例: thresh=2 で 3回出たら、3 - 2 = 1回分のリピート
                    # 例: thresh=1 で 2回出たら、2 - 1 = 1回分のリピート
                    repeat_counts = counts[mask] - self.repeat_thresh[n-1]
                    ngram_cnt[b] += repeat_counts.sum().float() * self.repeat_weight[n-1]
        
        #print( "DEBUG: ngram_cnt.mean():", ngram_cnt.mean() )
        #print( "DEBUG: seq_len:", seq_len )
        penalty = - (ngram_cnt / seq_len).clamp(max=1.0)
        
        #penalty = - torch.clamp(torch.pow(2.0, ngram_cnt - 1.0) / seq_len, max=1.0)

        #normalized_cnt = ngram_cnt / 10.0  
        ## 2. Sigmoid関数で0〜1の範囲に圧縮し、それをペナルティ係数とする
        ##    これにより、ngram_cntが増えてもペナルティは最大20(負の方向)に収束する
        #penalty = torch.sigmoid(normalized_cnt * 5 - 3) # ハイパーパラメータは調整可能
        #penalty_factor = torch.sigmoid(normalized_cnt * 5 - 3) # ハイパーパラメータは調整可能
        #penalty = - penalty_factor * 20

        #penalty = - torch.tanh(ngram_cnt / 3.0)

        #penalty = - torch.pow(ngram_cnt, 2.0) / seq_len * 5.0

        #penalty = - torch.nn.functional.softplus(torch.pow(1.5, ngram_cnt) - 1.0) / seq_len

        #penalty = - (torch.expm1(ngram_cnt / 3.0)) / seq_len * 20.0
        
        #penalty = - torch.clamp(torch.pow(2.0, ngram_cnt - 1.0) / seq_len, max=1.0) * 20

        #penalty = - torch.pow(2.0, ngram_cnt - 1.0) / seq_len * 20
        
        #return penalty[:, None].expand(-1, seq_len)
        return penalty[:, None]


    def calc_sep_eos_reward( self, preds ):

        eos_pattern = torch.tensor( [ 2 ], device = preds.device)
        sep_pattern = torch.tensor( [ 102 ], device = preds.device )
        eos_eos_pattern = torch.tensor( [2, 2], device = preds.device )
        sep_eos_pattern = torch.tensor( [102, 2 ], device = preds.device )
        target_pattern = torch.tensor([102, 2, 2], device = preds.device)
        pattern_eos_len = eos_pattern.shape[0]
        pattern_sep_len = sep_pattern.shape[0]
        pattern_eos_eos_len = eos_eos_pattern.shape[0]
        pattern_sep_eos_len = sep_eos_pattern.shape[0]
        pattern_target_len = target_pattern.shape[0]

        # 窓（スライディングウィンドウ）を使って3要素ずつ切り出して比較
        # 2次元テンソルを (bsz, seq_len - pattern_len + 1, pattern_len) にする
        windows_eos = preds.unfold(1, pattern_eos_len, 1)
        windows_sep = preds.unfold(1, pattern_sep_len, 1)
        windows_eos_eos = preds.unfold(1, pattern_eos_eos_len, 1)
        windows_sep_eos = preds.unfold(1, pattern_sep_eos_len, 1)
        windows_target = preds.unfold(1, pattern_target_len, 1)

        
        # windowsの形状は (2, 95, 3) 
        # 各位置の3要素がターゲットと一致するか判定
        matches_eos = (windows_eos == eos_pattern).all(dim=2)
        matches_sep = (windows_sep == sep_pattern).all(dim=2)
        matches_eos_eos = (windows_eos_eos == eos_eos_pattern).all(dim=2)
        matches_sep_eos = (windows_sep_eos == sep_eos_pattern).all(dim=2)
        matches_target = (windows_target == target_pattern).all(dim=2)

        
        # マッチした数をカウント
        count_eos = matches_eos.sum(1)
        count_sep = matches_sep.sum(1)
        count_eos_eos = matches_eos_eos.sum(1)
        count_sep_eos = matches_sep_eos.sum(1)
        count_target = matches_target.sum(1)
        
        reward_eos = ( count_eos == 1 ).float()
        reward_sep = 20 * ( count_sep == 1 ).float()
        reward_eos_eos = ( count_eos_eos == 1 ).float()
        reward_sep_eos = 20 * ( count_sep_eos == 1 ).float()
        reward_target = 20 * ( count_target == 1 ).float()
        not_reward_target = -10 * ( count_target != 1 ).float()
        not_reward_sep =  -10 * ( count_sep != 1 ).float()
        not_reward_eos_eos = - 10 * ( count_eos_eos != 1 ).float()
        reward = (reward_eos + reward_sep + reward_eos_eos + reward_sep_eos \
                  + reward_target + not_reward_target + not_reward_sep + not_reward_eos_eos ) / 6.0
        
        #print(f"出現回数: {eos_reward}")
        #print( "DEBUG: eos_reward.size():", eos_reward.size() )
        
        return reward[:, None]
        
    def forward(self, b2_preds, b2_targets, b2_imgs2,  sources=None, masks=None):
        """
        outputs: batch x len x d_model
        targets: batch x len
        sources: batch x len
        masks:   batch x len
        """
        self.device = b2_preds.device
        # input to device
        targets = b2_targets.to(self.device)
        bsz, seq_len = b2_preds.size()
        eps = 1e-8

        if self.reward_t == 'ordinary':
            reward_ord = self.compute_reward(b2_preds, b2_targets, b2_imgs2, sources)   #  bsz
            reward_repeat = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
            reward_length = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
            reward_unr = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
        elif self.reward_t == "ord+rep":
            reward_ord = self._compute_reward_ord(b2_preds, b2_targets, b2_imgs2, sources)  # bsz * seq_len
            reward_repeat = self.calc_ngram_repeat_fast( b2_preds ) # bsz * seq_len
            reward_length = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
            reward_unr = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
        elif self.reward_t == "ord+rep+len":
            reward_ord, reward_ord2 = self._compute_reward_ord(b2_preds, b2_targets, b2_imgs2, sources)  # bsz * seq_len
            reward_repeat = self.calc_ngram_repeat_fast( b2_preds ) # bsz * seq_len
            reward_length = self.compute_length_reward( b2_preds, b2_targets ) # bsz * seq_len
            reward_unr = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
        elif self.reward_t == 'ord+len':
            reward_ord = self._compute_reward_ord(b2_preds, b2_targets, b2_imgs2, sources)  # bsz * seq_len
            reward_repeat = self.calc_ngram_repeat_fast( b2_preds ) # bsz * seq_len
            reward_length = self.compute_length_reward( b2_preds, b2_targets ) # bsz * seq_len
            reward_unr = torch.zeros( ( bsz ),  device = preds.device, dtype=torch.float ) # bsz * seq_len
        elif self.reward_t == 'ord+rep+len+unr':
            reward_ord, reward_ord2 = self._compute_reward_ord(b2_preds, b2_targets, b2_imgs2, sources)  # bsz * seq_len
            reward_repeat = self.calc_ngram_repeat_fast( b2_preds ) # bsz * seq_len
            reward_length = self.compute_length_reward( b2_preds, b2_targets ) # bsz * seq_len
            reward_unr = self.unique_ngram_ratio(b2_preds)
            reward_eos = self.calc_sep_eos_reward(b2_preds)
        
        ## apply mask
        #if masks is not None:
        #    masks = masks.to(self.device)
        #    probs, targets = probs[masks], targets[masks]
        #    # outputs, targets = outputs[masks], targets[masks]
        #    reward, preds = reward[masks], preds[masks]
       
        #print(f'loss: {loss.item():.3f} | reward: {reward:.3f}')    
        
        return reward_ord, reward_ord2, reward_repeat, reward_length, reward_unr, reward_eos

In [ ]:
print( eos_token_id )
print( sep_token_id )

### CaptioningTransformer

In [ ]:
def logsumexp(x, dim=1):
    return torch.logsumexp(x.float(), dim=dim).type_as(x)

class DynamicCRF(nn.Module):
    def __init__(self, num_embedding, low_rank=32, beam_size=64, crf_coef=1.0, temp = 0.5, top_p = 0.9, top_k = 50, 
                 num_samples = 16, ref_t = False ):
        super().__init__()

        #low_rank = num_embedding
        self.E1 = nn.Embedding(num_embedding, low_rank)
        self.E2 = nn.Embedding(num_embedding, low_rank)

        self.vocb = num_embedding
        self.rank = low_rank
        self.beam = beam_size
        self.crf_coef = crf_coef
        self.temp = temp
        self.top_p = top_p
        self.top_k = top_k
        self.num_samples = num_samples
        self.ref_t = ref_t
        
    def extra_repr(self):
        return "vocab_size={}, low_rank={}, beam_size={}".format(
            self.vocb, self.rank, self.beam)

    def forward(self, emissions, top_logits, top_indices, targets, masks, beam=None):
        numerator = self._compute_score(emissions, targets, masks)
        denominator = self._compute_normalizer(emissions, targets, masks, beam )
        beam_probs = self._compute_normalizer2(top_logits, top_indices, targets, masks, beam)

        return numerator - denominator, beam_probs
    
    def forward_decoder(self, emissions, masks=None, beam=None):
        return self._viterbi_decode(emissions, masks, beam)

    def _compute_score(self, emissions, targets, masks=None):
        batch_size, seq_len = targets.size()

        emission_scores = emissions.gather(2, targets[:, :, None])[:, :, 0]  # B x T
        transition_scores = (self.E1(targets[:, :-1]) * self.E2(targets[:, 1:])).sum(2)
       
        scores = emission_scores
        scores[:, 1:] += transition_scores
        
        if masks is not None:
            scores = scores * masks.type_as(scores)

        return scores.sum(-1)
        
    def _compute_normalizer(self, emissions, targets=None, masks=None, beam=None):

        eps = 1e-8
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        if targets is not None:
            #_emissions = emissions.scatter(2, targets[:, :, None], np.float('inf'))
            _emissions = emissions.scatter(2, targets[:, :, None], float('inf'))
            beam_targets = _emissions.topk(beam, 2)[1]
            beam_emission_scores = emissions.gather(2, beam_targets)
        else:
            beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D; position i - 1, previous step.
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D; position i, current step.
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam)

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        for i in range(1, seq_len):
            next_score = score[:, :, None] + beam_transition_matrix[:, i-1]
            next_score = logsumexp(next_score, dim=1) + beam_emission_scores[:, i]
            if masks is not None:
                score = torch.where(masks[:, i:i+1], next_score, score)
            else:
                score = next_score

        return logsumexp(score, dim=1)
    '''
    def _compute_grpo_samples(self, beam_emission_scores, beam_transition_matrix, beam_targets, targets=None, masks=None, beam=None):
        
        eps = 1e-8
        device = beam_emission_scores.device

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = beam_emission_scores.size()[:2]

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []
        traj_scores2 = []
        traj_tokens2 = []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        _score2 = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1 ) #B * self.num_samples * K

        for i in range(1, seq_len):
            traj_scores.append(score) # bsz * beam
            traj_scores2.append( _score2 )
            _score2 = _score2[:,:,:,None] + beam_transition_matrix[:, i-1,None,:,:].expand( -1, self.num_samples,-1,-1) 
                    # bsz, self.num_samples, bema, beam

            # greedy selection
            #_score, _index = _score.max(dim=1) # bsz, beam     bsz, beam 

            ## multinomial selection
            B, N, C, W = _score2.shape
            flat_score = _score2.permute(0, 3, 1, 2).reshape(-1, C)
            #flat_score = torch.clamp( flat_score, min = -100, max = 100 )

            # --- 高速版 Top-K (行列演算のみ) ---
            #top_k = 50 
            if self.top_k > 0:
                # 各行の上位k番目の値を取得
                top_k_values, _ = torch.topk(logits, self.top_k, dim=-1)
                # k番目の値より小さいロジットを一括で -inf に置換
                min_values = top_k_values[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
            # ---------------------------------

            probs = F.softmax(logits, dim=-1)
            
            # 安全策: 全てが -Inf になった場合の回避
            if torch.isnan(probs).any():
                probs = torch.ones_like(probs) / probs.size(-1)

            _index_flat = torch.multinomial(probs, num_samples=1, replacement=True)  

            _score_flat = torch.gather(flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, W, self.num_samples).transpose(1,2)
            _score2 = _score_flat.view(B, W, self.num_samples).transpose(1,2)

            _score2 = _score2 + beam_emission_scores[:, i][:,None,:].expand(-1,self.num_samples,-1)

            score, index = _score2[:,0,:], _index2[:,0,:]
            traj_tokens.append(index)
            traj_tokens2.append( _index2 )

        all_scores = traj_scores2
        all_scores.append( _score2 )
        all_scores = torch.stack( all_scores, dim = 0 ).transpose( 0, 1 ).to(device) #bsz, seq_len, beam, N
        beam_probs = F.softmax( all_scores.transpose( 2, 3 ), dim = 2 ) #bsz, seq_len, beam, N
        #beam_probs = F.softmax( all_scores.permute( 3, 0, 1, 2 ), dim = 3 ) #N, bsz, seq_len, beam

        ## now running the back-tracing and find the best
        #best_score, best_index = _score2.max(dim=2) # max( bsz, beam ), bsz, N
        #finalized_tokens.append(best_index[:, None, :]) #bsz,1, N
        #finalized_scores.append(best_score[:, None, :]) #bsz,1, N

        #for idx, scs in zip(reversed(traj_tokens2), reversed(traj_scores2)): # each of seq_len -1, bsz, beam, N 
        #    previous_index = finalized_tokens[-1]
        #    finalized_tokens.append(idx.gather(2, previous_index))
        #    finalized_scores.append(scs.gather(2, previous_index))

        # 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        # _index2 が (B, W, self.num_samples) の場合、適切にリシェイプ
        # ※ W=1, self.num_samples=16 のケースが多いです
        current_sampled_index = _index2[:, :, 0:1] # (B, W, 1) 

        # 3. 最初の要素として追加
        finalized_tokens.append(current_sampled_index) # (B, W, 1)
        finalized_scores.append(_score2.gather(2, current_sampled_index)) # (B, W, 1)

        # 4. 過去に遡ってパスを復元
        # traj_tokens2 はループ内で append された各ステップのサンプリング結果
        for idx_step, scs_step in zip(reversed(traj_tokens2), reversed(traj_scores2)):
            # 直前のステップで選ばれたインデックスを使って、その前のインデックスを引く
            previous_pointer = finalized_tokens[-1]
            finalized_tokens.append(idx_step.gather(2, previous_pointer))
            finalized_scores.append(scs_step.gather(2, previous_pointer))

        finalized_tokens.reverse() # seq_len, bsz, N
        sampled_beam_idx = torch.cat(finalized_tokens, 1) # seq_len, bsz, N
        finalized_tokens = beam_targets.gather(2, sampled_beam_idx)

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        #return beam_probs, sampled_beam_idx, finalized_tokens, finalized_scores 
        return beam_probs, sampled_beam_idx, finalized_tokens 
    '''

    def _compute_grpo_samples(self, beam_emission_scores, beam_transition_matrix, beam_targets, targets=None, masks=None, beam=None):
        eps = 1e-8
        device = beam_emission_scores.device

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = beam_emission_scores.size()[:2]

        # フィルタリング用のパラメータ設定 (config等から取得できるよう適宜調整してください)
        #top_k = 50  # 上位k個に絞る (0なら無効)
        #top_p = 0.9 # 累積確率pまでに絞る (1.0なら無効)

        traj_tokens, traj_scores = [], []
        traj_scores2 = []
        traj_tokens2 = []

        score = beam_emission_scores[:, 0]
        _score2 = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1 ).clone()
        # --- 多様性を強制するためのノイズ追加 ---
        # 各グループが異なる探索を始めるように、微小なノイズを加える
        # (1e-5 程度のノイズで十分です。順位が入れ替わるきっかけを作ります)
        noise = torch.randn_like(_score2) * 1e-5
        _score2 = _score2 + noise

        for i in range(1, seq_len):
            traj_scores.append(score)
            traj_scores2.append( _score2 )
            _score2 = _score2[:,:,:,None] + beam_transition_matrix[:, i-1,None,:,:].expand( -1, self.num_samples,-1,-1) 

            #B, N, C, W = _score2.shape
            #flat_score = _score2.permute(0, 3, 1, 2).reshape(-1, C)
            B, N, C, W = _score2.shape
            flat_score = _score2.permute(0, 3, 1, 2).reshape(-1, C)
            #logits = flat_score / self.temp
            
            ## --- 高速版 Top-K (行列演算のみ) ---
            ##top_k = 50 
            #if self.top_k > 0:
            #    # 各行の上位k番目の値を取得
            #    top_k_values, _ = torch.topk(logits, self.top_k, dim=-1)
            #    # k番目の値より小さいロジットを一括で -inf に置換
            #    min_values = top_k_values[:, -1].unsqueeze(-1)
            #    logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
            ## ---------------------------------

            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp
            
            # 1. まず Top-K で上位50個に絞る
            # ※ここで全語彙に対する Top-K を行い、小さなテンソルに切り出します
            top_k_val = 50 
            top_k_logits, top_k_indices = torch.topk(logits, top_k_val, dim=-1)

            # 2. Top-P (Nucleus) filtering
            if self.top_p < 1.0:
                # Top-Kで絞った「50個」の中でソート
                sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True)
                
                # 累積確率を計算
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                # 除去対象のマスクを作成
                sorted_indices_to_remove = cumulative_probs > self.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                # 【重要】top_k_logits と同じ形状 (サイズ50) のマスクを作成する
                # ここを zeros_like(logits) ではなく zeros_like(top_k_logits) に修正
                indices_to_remove_k = torch.zeros_like(top_k_logits, dtype=torch.bool).scatter_(
                    dim=-1, 
                    index=sorted_indices, 
                    src=sorted_indices_to_remove
                )

                # サイズ50のテンソルに対してマスクを適用
                top_k_logits[indices_to_remove_k] = -float('Inf')

            # 3. 全語彙の logits を一旦すべて -inf にして、生き残った top_k_logits だけを戻す
            new_logits = torch.full_like(logits, float('-inf'))
            new_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_logits)

            logits = new_logits
            # --- Top-K / Top-P Filtering 終了 ---

            probs = F.softmax(logits, dim=-1)
            
            # 安全策: 全てが -Inf になった場合の回避
            if torch.isnan(probs).any():
                probs = torch.ones_like(probs) / probs.size(-1)

            _index_flat = torch.multinomial(probs, num_samples=1, replacement=True)  

            _score_flat = torch.gather(flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, W, self.num_samples).transpose(1,2)
            _score2 = _score_flat.view(B, W, self.num_samples).transpose(1,2)

            _score2 = _score2 + beam_emission_scores[:, i][:,None,:].expand(-1,self.num_samples,-1)

            score, index = _score2[:,0,:], _index2[:,0,:]
            traj_tokens.append(index)
            traj_tokens2.append( _index2 )

        # --- 以下、元のバックトレーシング処理 ---
        all_scores = traj_scores2
        all_scores.append( _score2 )
        all_scores = torch.stack( all_scores, dim = 0 ).transpose( 0, 1 ).to(device)
        beam_probs = F.softmax( all_scores.transpose( 2, 3 ), dim = 2 )

        #best_score, best_index = _score2.max(dim=2)
        #finalized_tokens.append(best_index[:, None, :])
        #finalized_scores.append(best_score[:, None, :])

        # --- 修正後 ---
        # 最後のステップでサンプリングされたインデックスをそのまま使う
        # _index2 は (Batch, Group, Beam) の形状のはずです        
        # 最後のステップでサンプリングされたインデックスをそのまま使う
        # last_sampled_index (B, N, W) と想定
        last_sampled_index = _index2 
        
        # 1. finalized_tokens への追加 (B, 1, N, W にならないよう注意)
        # もし _score2 が (B, N, W) なら、index も (B, N, 1) などにする必要があります
        # ここでは gather 用に最後の次元を 1 にした index を作ります
        
        # last_sampled_index が (Batch, Group, Beam) の場合、
        # 通常バックトレーシングで必要なのは「どのビームを選んだか」というインデックスです
        # ここでは _score2 の最後の次元(2)に対して gather します

        # --- 修正：バックトレーシングの開始 ---
        # 1. まずリストを空にする（重要！）
        finalized_tokens = []
        finalized_scores = []

        # 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        # _index2 が (B, W, self.num_samples) の場合、適切にリシェイプ
        # ※ W=1, self.num_samples=16 のケースが多いです
        current_sampled_index = _index2[:, :, 0:1] # (B, W, 1) 

        # 3. 最初の要素として追加
        finalized_tokens.append(current_sampled_index) # (B, W, 1)
        finalized_scores.append(_score2.gather(2, current_sampled_index)) # (B, W, 1)

        # 4. 過去に遡ってパスを復元
        # traj_tokens2 はループ内で append された各ステップのサンプリング結果
        for idx_step, scs_step in zip(reversed(traj_tokens2), reversed(traj_scores2)):
            # 直前のステップで選ばれたインデックスを使って、その前のインデックスを引く
            previous_pointer = finalized_tokens[-1]
            finalized_tokens.append(idx_step.gather(2, previous_pointer))
            finalized_scores.append(scs_step.gather(2, previous_pointer))

        # 5. 逆順にして結合
        # 5. 逆順にして結合 (この時点では [seq_len, B, Group, 1] のリスト)
        finalized_tokens.reverse()

        sampled_beam_idx = torch.cat(finalized_tokens, 2) # [B, G, S]

        # 2. beam_targets [B, S, Beam] と合わせるために transpose
        # [B, G, S] -> [B, S, G] にして、各ステップ(S)ごとのビーム(G)を選択
        sampled_beam_idx = sampled_beam_idx.permute(0, 2, 1) # [B, S, G]

        # 3. gather 実行 (第2次元[Beam方向]から、サンプリングしたインデックスを抽出)
        finalized_tokens = beam_targets.gather(2, sampled_beam_idx) # [B, S, G]
        
        #finalized_scores.reverse()
        #finalized_scores = torch.cat(finalized_scores, 1)
        #finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]
        
        return beam_probs, sampled_beam_idx, finalized_tokens
    '''
    def _viterbi_decode(self, emissions, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        #print( "score size:", score.size() )
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

        for i in range(1, seq_len):
            traj_scores.append(score)
            _score = score[:, :, None] + beam_transition_matrix[:, i-1] # bsz, beam, beam
            _score, _index = _score.max(dim=1) # bsz, beam     bsz, beam 
            _score = _score + beam_emission_scores[:, i] # bsz, beam

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        # now running the back-tracing and find the best
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])

        for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)):
            previous_index = finalized_tokens[-1]
            finalized_tokens.append(idx.gather(1, previous_index))
            finalized_scores.append(scs.gather(1, previous_index))

        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        return finalized_scores, finalized_tokens

   
    '''
    def _compute_many_values(self, emissions, targets, top_indices = None, masks=None, beam=None):

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        
        if top_indices == None:
            beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        else:
            beam_emission_scores = torch.gather( emissions, -1, top_indices )
            beam_targets = top_indices
        
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        if not self.ref_t:
            traj_tokens, traj_scores = [], []
            finalized_tokens, finalized_scores = [], []

            # compute the normalizer in the log-space
            score = beam_emission_scores[:, 0]  # B x K
            dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

            for i in range(1, seq_len):
                traj_scores.append(score)
                _score = score[:, :, None] + beam_transition_matrix[:, i-1] # bsz, beam, beam
                _score, _index = _score.max(dim=1) # bsz, beam     bsz, beam  step i-1 における 256 → 256 の max から 256 への遷移確率と 
                                                # 256 → 256 の前の 256 の max のインデックストークン
                                                # index b * 256 の 位置が i の token で、値が i-1 のtoken   

                _score = _score + beam_emission_scores[:, i] # bsz, beam i における 256 の遷移確率ではない確率を加える。i における 256 の全確率。

                if masks is not None:
                    score = torch.where(masks[:, i: i+1], _score, score)
                    index = torch.where(masks[:, i: i+1], _index, dummy)
                else:
                    score, index = _score, _index
                traj_tokens.append(index)

            all_scores = traj_scores
            all_scores.append( score )
            all_scores = torch.stack( all_scores, dim = 0 ).transpose( 0, 1 )
        
            ## now running the back-tracing and find the best
            best_score, best_index = score.max(dim=1)
            finalized_tokens.append(best_index[:, None]) # 時刻 T における b*256 の確率最大の token
            finalized_scores.append(best_score[:, None]) #時刻 T における b*256 の確率最大の score


            for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)): #idx,scs は、反転時刻 i と i-1における b * 256のトークンと確率
                previous_index = finalized_tokens[-1] # 時刻 Tなど 求めたいトークンと確率の一個後 における token　b * 1
                finalized_tokens.append(idx.gather(1, previous_index)) # 時刻 一個後iのトークン previou_index に至るための時刻i-1 のトークン
                                                                    # b* 256 の token から b * 1 の previous_idnex token で gather
                finalized_scores.append(scs.gather(1, previous_index)) # 時刻一個後 i のトークンに至るための時刻 i-1 の確率

            finalized_tokens.reverse()
            finalized_tokens = torch.cat(finalized_tokens, 1)
            finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

            finalized_scores.reverse()
            finalized_scores = torch.cat(finalized_scores, 1)
            finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]
        
            if self.crf_coef != 0.0:
                numerator = self._compute_score(emissions, targets)
                denominator = self._compute_normalizer(emissions, targets)
                crf_loss = - ( numerator - denominator ).mean() / seq_len
            else:
                crf_loss = torch.zeros( (1), device = emissions.device, dtype = torch.float )
        
        top_probs, sampled_beam_idx, b_finalized_tokens = \
            self._compute_grpo_samples(beam_emission_scores, beam_transition_matrix, beam_targets)

        if not self.ref_t:
            return finalized_scores, finalized_tokens, top_probs, beam_targets, crf_loss, sampled_beam_idx, b_finalized_tokens
        else:
            return top_probs[:,:,:,0]

In [ ]:
class TopLayer(nn.Module):
    def __init__(self, vocab_size, embed_dim, crf_low_rank, crf_beam_size, dropout, padding_idx,
                crf_coef = 1.0, temp = 0.5, top_k = 0.9, top_p = 50, num_samples = 16, ref_t = False ):
        super(TopLayer, self).__init__()

        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.dropout = dropout
        self.padding_idx = padding_idx
        print( "in TopLayer:" )
        self.crf_layer = DynamicCRF(num_embedding = vocab_size, low_rank = crf_low_rank, beam_size = crf_beam_size, 
                                    crf_coef=crf_coef, temp=temp, top_p = top_p, top_k = top_k, num_samples= num_samples, ref_t = ref_t )

        #self.one_more_layer_norm = nn.LayerNorm(embed_dim)
        #self.tgt_word_prj = nn.Linear(self.embed_dim, self.vocab_size)
        ## gae 学習用
        #self.linear_critical = nn.Linear(crf_beam_size, 1 )

    def forward(self, src_representation, top_logits, top_indices, src_input, tgt_input, is_training ):
        '''
            src_representation : bsz x seqlen x embed_dim
            src_input : bsz x seqlen
            tgt_input : bsz x seqlen
        '''
        #assert src_input.size() == tgt_input.size()

        src_input = src_input.transpose(0, 1) # src_len x bsz
        #seqlen, bsz = src_input.size()
        seqlen, bsz = src_input.shape[:2]

        src_representation = F.dropout(src_representation, p=self.dropout, training=is_training)
        src_representation = src_representation.transpose(0, 1) # seqlen x bsz x embed_dim

        src = src_representation

        #emissions = self.tgt_word_prj(src.contiguous().view(-1, self.embed_dim)).view(seqlen, bsz, self.vocab_size)
        emissions = src_representation
        #log_probs = torch.log_softmax(emissions, -1)
        #assert log_probs.size() == torch.Size([seqlen, bsz, self.vocab_size])

        emissions = emissions.transpose(0, 1) # [bsz x src_len x vocab_size]
        #emission_mask = ~tgt_input.eq(self.padding_idx) # [bsz x src_len] #pad のところは 0 padでないところが 1
        emission_mask = torch.ones_like( tgt_input, dtype=torch.bool ) #全部　pad でないとして 1
        batch_crf_loss, top_probs = self.crf_layer(emissions, top_logits, top_indices, tgt_input, emission_mask) # [bsz]
        #critical_value = self.linear_critical( top_probs )
        #critical_value = torch.zeros( ( 1,1,1) )
        batch_crf_loss = - batch_crf_loss
        assert batch_crf_loss.size() == torch.Size([bsz])
        return batch_crf_loss, top_probs

    def decoding(self, src_representation, src_input):
        '''
            src_representation : bsz x seqlen x embed_dim
            src_input : bsz x seqlen
            tgt_input : bsz x seqlen
        '''
        src_input = src_input.transpose(0, 1) # src_len x bsz
        seqlen, bsz = src_input.size()

        src_representation = src_representation.transpose(0, 1) # seqlen x bsz x embed_dim
        src = src_representation

        emissions = self.tgt_word_prj(src.contiguous().view(-1, self.embed_dim)).view(seqlen, bsz, self.vocab_size)

        emissions = emissions.transpose(0, 1) # [bsz, seqlen, vocab_size]
        _, finalized_tokens = self.crf_layer.forward_decoder(emissions)
        assert finalized_tokens.size() == torch.Size([bsz, seqlen])
        return finalized_tokens


In [ ]:
class CaptioningTransformer(nn.Module):
    
    #CaptioningTransformerのコンストラクタ
    #dim_embedding  : 埋め込み次元
    #dim_feedforward: FNNの中間特徴次元
    #num_heads      : マルチヘッドアテンションのヘッド数
    #num_layers     : Transformerデコーダ層の数
    #vocab_size     : 辞書の次元
    #null_index     : NULLのID
    #dropout        : ドロップアウト確率
    
    def __init__(self, img_size: int,  dim_embedding: int, length_max: int, vocab_size: int, tokenizer, dropout: float = 0.0, \
                 pad_token_id: int=0, use_repeat_logits_half=False, crf_coef = 1.0, temp=0.5, top_p = 0.9, top_k = 50,
                 num_samples=16, ref_t = False):
        super().__init__()

        #CLIP
        model_id = "openai/clip-vit-large-patch14-336"
        self.clip_model = CLIPVisionModel.from_pretrained(model_id )
        memory = self.clip_model( torch.randn( 1, 3, 336, 336 ) )
        memory = memory.last_hidden_state
        img_length = memory.size(1)
        clip_dim = memory.size(2)
        self.connector_pool = nn.AdaptiveAvgPool1d(length_max - 1 )
        self.connector_ln = nn.LayerNorm( clip_dim )
        self.connector_linear1 = nn.Linear( clip_dim, dim_embedding )
        self.connector_gleu = nn.GELU()
        self.connector_linear2 = nn.Linear( dim_embedding, dim_embedding )

       
        # Connector
        self.connector_pool = nn.AdaptiveAvgPool1d(length_max - 1 )
       # Down Sampling
        cls_token = memory[:, :1, :] # (bsz, 1, 1024)
        patch_tokens = memory[:, 1:, :] # (bsz, 576, 1024)
        # パッチ部分を 576 -> 96 に圧縮
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 1024, 576)
        patch_tokens = self.connector_pool(patch_tokens)
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 96, 1024)
        # CLSと結合して合計 97 トークンにする
        memory = torch.cat([cls_token, patch_tokens], dim=1) # (bsz, 97, 1024)

        self.pos_emb = PositionalEmbedding( dim_embedding )

        model_id = "google-bert/bert-large-uncased"
        self.bert = BertModel.from_pretrained( model_id )

        ## 単語出力分布計算
        self.ln_outputs = nn.LayerNorm( dim_embedding )
        self.linear = nn.Linear(dim_embedding, vocab_size)

        crf_low_rank = 32
        crf_beam_size = 256
        self.crf_beam_size = crf_beam_size
        top_dropout = 0.0
        tgt_padding_idx = tokenizer.pad_token_id
        print( "initialize self.toplayer" )
        if ref_t:
            num_samples = 1
        self.ref_t = ref_t
        self.toplayer = TopLayer( vocab_size, dim_embedding, crf_low_rank, crf_beam_size, top_dropout, 
                                  tgt_padding_idx, crf_coef = crf_coef, temp=temp, top_p = top_p, top_k = top_k,
                                  num_samples=num_samples, ref_t = ref_t )

        self.dim_embedding = dim_embedding
        self.use_repeat_logits_half = use_repeat_logits_half


    def mlp_connector(self, memory ):

        cls_token = memory[:, :1, :] # (bsz, 1, 1024)
        patch_tokens = memory[:, 1:, :] # (bsz, 576, 1024)

        # パッチ部分を 576 -> 96 に圧縮
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 1024, 576)
        patch_tokens = self.connector_pool(patch_tokens)
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 96, 1024)

        # CLSと結合して合計 97 トークンにする
        memory = torch.cat([cls_token, patch_tokens], dim=1) # (bsz, 97, 1024)

        memory = self.connector_ln( memory )
        memory = self.connector_linear1( memory )
        memory = self.connector_gleu( memory )
        memory = self.connector_linear2( memory )
        
        return memory

    def forward(self, images: torch.Tensor, targets: torch.Tensor, top_indices = None ):

        self.device = images.device
        
        memory = self.clip_model( images ).last_hidden_state
        memory = self.mlp_connector( memory )
        memory += self.pos_emb( memory )
        
        outputs = self.bert( inputs_embeds = memory ).last_hidden_state
        outputs = self.ln_outputs( outputs )
        emissions = self.linear( outputs )

        if self.use_repeat_logits_half == True:
            emissions = repeat_logits_half( emissions )

        if not self.ref_t:
            finalized_scores, finalized_tokens, top_probs, top_indices, crf_loss, sampled_beam_idx, b_finalized_tokens  = \
                self.toplayer.crf_layer._compute_many_values(emissions, targets, top_indices)
            return finalized_scores, finalized_tokens, top_probs, top_indices, \
                 crf_loss, emissions, sampled_beam_idx, b_finalized_tokens
        else:
            top_probs = self.toplayer.crf_layer._compute_many_values(emissions, targets, top_indices)
            return top_probs

    def repeat_logits_half(self, emissions ):
        
        penalty = 1.2
        scores, preds = torch.max( emissions, 2 )
        masks = emissions == scores[:,:,None]
        masks = masks.permute( 1, 0, 2 )
        new_mask = torch.zeros( (  masks.size(1), masks.size(2)), device = emissions.device, dtype=torch.bool )
        new_masks = torch.zeros( ( masks.size(0), masks.size(1), masks.size(2)), device = emissions.device, dtype=torch.bool )
        for i, mask in enumerate( masks ):
            new_mask = torch.logical_or( mask,  new_mask  )
            new_masks[i] = new_mask
        new_masks = new_masks.transpose(0,1)
        first_true_mask = ( new_masks.int().cumsum(dim = 1 ) == 1 ) & new_masks
        new_masks = new_masks & ( ~first_true_mask )

        p_masks = emissions > 0
        m_masks = emissions < 0
        p_new_masks = p_masks & new_masks
        m_new_masks = p_masks & new_masks
        emissions2 = emissions.clone()
        emissions2[p_new_masks] = emissions[p_new_masks] / penalty
        emissions2[m_new_masks] = emissions2[m_new_masks] * penalty

        return emissions2

In [ ]:
class MyDataset(Dataset):
    def __init__(self, file_path: str, img_directory: str, transforms, transforms2, tokenizer, length_max = None ) -> None:
        super().__init__()
        self.img_directory = img_directory
        self.transforms = transforms
        self.transforms2 = transforms2 
        # TODO: fix to original data
        #画像の前処理
        self.img_file = []
        self.tokens = []
        #vocab_size = len( tokenizer )
        #c1 = torch.zeros( ( vocab_size ) )
        #c2 = torch.zeros( ( vocab_size, vocab_size ) )
        if length_max == None:
            self.length_max = 0
        else:
            self.length_max = length_max
        length_sum = 0

        with open( file_path, 'rb') as f:
            data = pickle.load(f)
        for i, line_data in enumerate( data ):
            if i % 100000 == 0:
                print( "i:", i )
            self.img_file.append( line_data['img_file'] )
            id_tokens = line_data['id_tokens']
            id_tokens.append( eos_token_id )
            id_tokens.append( eos_token_id )
            length_sum += len( id_tokens )
            if length_max != None:
                id_tokens = torch.tensor( id_tokens )[:self.length_max]
            else:
                if self.length_max < len( id_tokens ):
                    self.length_max = len( id_tokens )
                id_tokens = torch.tensor( id_tokens )
            self.tokens.append( id_tokens )
        # w1, w2 を作る時は length_max = None　でお願いします。
        #    for i2 in range( len(id_tokens) ):
        #        if i2 == len( id_tokens ) - 1:
        #            c1[id_tokens[i2]] += 1
        #        else:
        #            c1[id_tokens[i2]] += 1
        #            c2[id_tokens[i2], id_tokens[i2+1] ] += 1
        '''
        c1avg = int( torch.sum( c1 ) / torch.sum( torch.ne( c1, 0 ).int()) )
        c2avg = int( torch.sum( torch.sum( c2, dim = 1 ), dim = 0 ) / torch.sum( torch.ne( c2, 0 ).int() ) )

        c1[0] = c1avg

        c2[:,0] = c2avg
        c2[0,:] = c2avg
        
        sumc1 = torch.sum( c1, dim = 0 )
        sumc2 = torch.sum( torch.sum( c2, dim = 1 ), dim = 0 )

        prob1 = c1 / sumc1
        prob2 = c2 / sumc2

        self.w1 = prob1 ** -0.4
        self.w1 = torch.nan_to_num( self.w1, nan = 0.0, posinf=0.0, neginf=0.0 )
        avg1 = torch.sum( self.w1, dim = 0 ) / torch.sum( torch.ne( self.w1, 0.0 ).int() )
        self.w1 = self.w1 / avg1

        self.w2 = prob2 ** -0.4
        self.w2 = torch.nan_to_num( self.w2, nan = 0.0, posinf=0.0, neginf=0.0 )
        avg2 = torch.sum( torch.sum( self.w2, dim = 1 ), dim = 0 ) / torch.sum( torch.ne( self.w2, 0.0 ).int() )
        self.w2 = self.w2 / avg2

        with open( "/mnt/ssd2/v7/w_unigrma.pkl", mode="wb" ) as f:
            pickle.dump( self.w1, f )

        with open( "/mnt/ssd2/v7/w_bigrma.pkl", mode="wb" ) as f:
            pickle.dump( self.w2, f )
        
        '''

        #with open( "/mnt/ssd2/v7/w_unigram.pkl", 'rb') as f:
        #    self.w1 = pickle.load(f)

        #with open( "/mnt/ssd2/v7/w_bigram.pkl", 'rb') as f:
        #    self.w2 = pickle.load(f)
        
        if length_max == None:
            print( "length max:", self.length_max )
            print( "avg length:", length_sum / len( self.tokens ) )
    
    # ここで取り出すデータを指定している
    def __getitem__(
        self,
        index: int
    ):
        tokens = self.tokens[index]
        img_file = self.img_file[index] + ".jpg"
        img_path = os.path.join( self.img_directory, img_file ) #index番目の画像のパスを取得
        img = Image.open(img_path) #PIL形式で画像を読み込み
        if img.mode != 'RGB':
            img = img.convert("RGB")
        img1 = self.transforms(img)
        img2 = self.transforms2(img)
        
        return img1, img2, tokens

    # この method がないと DataLoader を呼び出す際にエラーを吐かれる
    def __len__(self) -> int:
        return len(self.tokens)

    def length_max(self):
        return self.length_max

    #def w1(self):
    #    return self.w1

    #def w2(self):
    #    return self.w2

In [ ]:
def collate_func(batch: Sequence[Tuple[Union[torch.Tensor, str]]], pad_index, length_max ):
    imgs1, imgs2, tokens = zip(*batch)

    max_length = length_max
    #max_length = 0
    #for target in tokens:
    #    if max_length < len( target ):
    #        max_length = len( target )
    
    targets = []
    lengths = []
    for target in tokens:
        pad_len = max_length - len( target ) 
        #print( "target:", target )
        input2= F.pad( target, (0, pad_len), mode='constant', value = pad_index)
        targets.append( input2 )
        lengths.append( len( target ) )
    
    imgs1 = torch.stack( imgs1, dim = 0 )
    imgs2 = torch.stack( imgs2, dim = 0 )
    targets = torch.stack( targets, dim = 0 )
    lengths = torch.tensor( lengths, requires_grad = False  )

    return imgs1, imgs2, targets, lengths

In [ ]:
model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
sos_token_id = tokenizer.encode( [ "[unused0]" ] )[1]
eos_token_id = tokenizer.encode( [ "[unused1]" ] )[1]
a_token_id = tokenizer.encode( [ "a" ] )[1]
the_token_id = tokenizer.encode( [ "the" ] )[1]
and_token_id = tokenizer.encode( [ "and" ] )[1]
in_token_id = tokenizer.encode( [ "in" ] )[1]
we_token_id = tokenizer.encode( [ "we" ] )[1]
i_token_id = tokenizer.encode( [ "i" ] )[1]
he_token_id = tokenizer.encode( [ "he" ] )[1]
she_token_id = tokenizer.encode( [ "she" ] )[1]
it_token_id = tokenizer.encode( [ "it" ] )[1]
they_token_id = tokenizer.encode( [ "they" ] )[1]
period_token_id = tokenizer.encode( [ "." ] )[1]
comma_token_id = tokenizer.encode( [ "," ] )[1]
dbl_token_id = tokenizer.encode( [ '"' ] )[1]
sgl_token_id = tokenizer.encode( [ "'" ] )[1]


# 辞書サイズを保存
vocab_size = len( tokenizer )

# 画像のtransformsを定義
transforms = v2.Compose([
    v2.Resize((336, 336)),
    #v2.AutoAugment(),
    #v2.ToTensor(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    ## Coco データセット 2017 train の平均と標準偏差
    #v2.Normalize((0.456,0.427,0.401),(0.224,0.219,0.231) )
    ## ImageNetデータセットの平均と標準偏差
    #v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    # clip の preprocessor_config.json の平均と標準偏差
    v2.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
])

transforms2 = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

# v7 データセット
train_dataset = MyDataset( file_path="/mnt/ssd2/v7/data.pkl",
                           img_directory = "/mnt/ssd2/v7/img",
                           transforms=transforms, transforms2 = transforms2, tokenizer = tokenizer, length_max = 97 )

# Subset samplerの生成
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, 0.1, 0.1 )
    
# 学習時にランダムにサンプルするためのサンプラー
train_sampler = SubsetRandomSampler(train_set)

# DataLoaderを生成
collate_func_lambda = lambda x: collate_func(x, tokenizer.pad_token_id, length_max = 97 )

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=4,
                    num_workers=0,
                    sampler=test_set,
                    collate_fn=collate_func_lambda)


In [ ]:
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
#model = CaptioningTransformer( img_size = 336, dim_embedding=1024, length_max = 97, vocab_size=len(tokenizer),
#                 tokenizer=tokenizer, dropout=0.1).to(device)
#model = CaptioningTransformer( 336,
#    1024, 97, len( tokenizer ),
#    tokenizer, 0.0, pad_token_id = tokenizer.pad_token_id,
#    use_repeat_logits_half = False,
#    crf_coef = 0.0, temp=1.0 )
#model = CaptioningTransformer( 336,
#    1024, 97, len( tokenizer ),
#    tokenizer, 0.0, pad_token_id = tokenizer.pad_token_id,
#    use_repeat_logits_half = False,
#    crf_coef = 0.0, temp=0.710 )
##model.to(config.device)
#model.to(device)

temp = 0.8
top_p = 0.9
top_k = 50
num_samples = 16

model = CaptioningTransformer( 336,
    1024, 97, len( tokenizer ),
    tokenizer, 0.0, pad_token_id = tokenizer.pad_token_id,
    use_repeat_logits_half = False,
    crf_coef = 0.0, temp=0.8, top_p = top_p, 
    top_k = top_k, num_samples=num_samples )
model.to(device)

PATH = "/mnt/ssd2/model_rl_grpo_crf24_.pth"
#PATH = "./model/model_rl_grpo_crf34_108_12_.pth"
if os.path.isfile(PATH):
    checkpoint = torch.load(PATH, map_location=torch.device("cpu"))
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    print( "paramerters were loaded." )

compute_reward = ComputeReward(reward_t = "ord+rep+len+unr", decode_t = "no-pad", 
                          device = device, repeat_thresh = (3,2,2,2,2), 
                          repeat_weight = (1,1,1,1, 1), cider_coef = 1.0, rouge_coef = 1.0, 
                          clip_coef = 1.0, bert_coef = 1.0, use_amp = False )

#compute_reward = ComputeReward(reward_t = config.reward_t, decode_t = config.decode_t, device = config.device, 
#                               repeat_thresh = config.repeat_thresh, 
#                          repeat_weight = config.repeat_weight, cider_coef = config.cider_coef, rouge_coef = config.rouge_coef, 
#                          clip_coef = config.clip_coef, bert_coef = config.bert_coef, use_amp = config.use_amp )



In [ ]:
images = torch.randn( ( 2, 3, 336,336 ), device = device )
targets = torch.randint( 0, len( tokenizer ), size = ( 2, 97 ) )

finalized_scores, finalized_tokens, top_probss, top_indices, \
crf_loss, bert_logits, sampled_beam_idxs, b_finalized_tokens  = \
model( images, targets, top_indices = None )

print( finalized_tokens.size() )

In [ ]:
use_amp = False
test_num = 80
batch_size = 8
seq_len = 97

model_name = "distilbert-base-uncased"

pad_token_id = tokenizer.pad_token_id

my_decode = False
#my_decode = True

# Subset samplerの生成
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, 0.1, 0.1 )

test_set = test_set[:test_num]

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=batch_size,
                    num_workers=0,
                    sampler=test_set,
                    collate_fn=collate_func_lambda)

test_pr_coef = 1


transforms_inv = v2.Compose([
    v2.Normalize((-0.48145466/0.26862954, -0.4578275/0.26130258, -0.40821073/0.27577711), (1/0.26862954,1/0.26130258,1/0.27577711)),
    v2.ToPILImage()
])

# test
with tqdm(test_loader) as pbar:
    pbar.set_description(f'[テスト]')

    # 評価モード
    model.eval()

    test_ciders = deque()
    test_rouges = deque()
    test_clips = deque()
    test_berts = deque()
    n_iter = 0
    for k, (imgs, imgs2, captions, caption_lengths) in enumerate( pbar ):
        print( "k:", k )
        imgs = imgs.to(device)
        captions = captions.to(device)
        #caption_lengths = torch.tensor( caption_lengths ).to(config.device)
        
        with torch.no_grad():
            finalized_scores, finalized_tokens, top_probss, top_indices, \
            crf_loss, bert_logits, sampled_beam_idxs, b_finalized_tokens  = \
            model( imgs, captions, top_indices = None )
            hypo_ids = finalized_tokens
        
        n = 0
        hypo_sentence = []
        ref_sentence = []
        ref_imgs = []
        total_error = 0
        total_token_length = 0
        total_bleu = 0
        hypo_ids[hypo_ids == eos_token_id] = pad_token_id
        decoded = tokenizer.batch_decode(hypo_ids, skip_special_tokens=False)
        preds_str = [s.replace("[PAD]", "").replace("[unused1]", "").strip() for s in decoded]
        preds_str2 = tokenizer.batch_decode(hypo_ids, skip_special_tokens = True )
        captions[captions == eos_token_id] = pad_token_id
        decoded = tokenizer.batch_decode(captions, skip_special_tokens=False)
        targets_str = [s.replace("[PAD]", "").replace("[unused1]", "").strip() for s in decoded]  

        hypo_tokens = tokenizer.tokenize( preds_str[0] )
        ref_tokens = tokenizer.tokenize( targets_str[0] )

        pred_dict = { str(i): [item] for i, item in enumerate( preds_str)}
        target_dict = { str(i): [item] for i, item in enumerate( targets_str)}
        '''
        with torch.no_grad():
            #print( "DEBUG: pred_dict.keys():", pred_dict.keys() )
            #print( "DEBUG: target_dict.keys():", target_dict.keys() )
            avg_cider, scores = compute_reward.scorer.compute_score(target_dict, pred_dict) # cider の計算
            rouge_scores = [compute_reward.rougeL.score(target, pred)['rougeL'][0] for pred, target in zip(preds_str, targets_str)]
            avg_rouge = sum( rouge_scores ) / len( rouge_scores )
        with autocast(str(device),enabled=use_amp):
            with torch.no_grad():
                #reward_clip = compute_reward_with_metrics( preds_str2, imgs2 )
                processed = compute_reward.metric.processor(text=preds_str2, images=imgs2, return_tensors="pt", padding=True, \
                                truncation=True, max_length=77, do_resize=False, do_rescale=False ).to(device)
                outputs = compute_reward.metric.model(**processed)
                # 特徴量の正規化
                image_features = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
                text_features = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
                individual_scores = torch.clamp( (image_features.to(device) * \
                    text_features.to(device)).sum(axis=-1), min=0)
                clip_scores = individual_scores[:,None].expand( -1, seq_len )
                clip_score = clip_scores.mean()
                #bert_scores = compute_reward.bert.compute(predictions=preds_str, references=targets_str, model_type=model_name, \
                #                                  lang='en',  device=device)['f1']
                model_name = 'distilbert-base-uncased' 
                bert_scores = compute_reward.bert.compute(
                predictions=preds_str, 
                    references=targets_str,
                    model_type=model_name,
                    use_fast_tokenizer=True, 
                    lang='en', 
                    device=device,
                    batch_size=batch_size,  # メモリ許容範囲で大きく設定
                    rescale_with_baseline=False
                    )['f1']
                bert_score = sum( bert_scores ) / len( bert_scores )

        '''
        avg_cider, scores = compute_reward.scorer.compute_score(target_dict, pred_dict) # cider の計算        
        rouge_scores = [compute_reward.rougeL.score(target, pred)['rougeL'][0] for pred, target in zip(preds_str, targets_str)]
        avg_rouge = sum( rouge_scores ) / len( rouge_scores )
        with torch.no_grad():
            #reward_clip = compute_reward_with_metrics( preds_str2, imgs2 )
            processed = compute_reward.metric.processor(text=preds_str2, images=imgs2, return_tensors="pt", padding=True, \
                                              truncation=True, max_length=77, do_resize=False, do_rescale=False ).to(device)
            outputs = compute_reward.metric.model(**processed)
            # 特徴量の正規化
            image_features = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
            text_features = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
            individual_scores = torch.clamp( (image_features.to(device) * \
                                              text_features.to(device)).sum(axis=-1), min=0)
            clip_scores = individual_scores[:,None].expand( -1, seq_len )
            clip_score = clip_scores.mean()
            #bert_scores = compute_reward.bert.compute(predictions=preds_str, references=targets_str, model_type=model_name, \
            #                                  lang='en',  device=config.device)['f1']
            model_name = 'distilbert-base-uncased' 
            bert_scores = compute_reward.bert.compute(
                predictions=preds_str, 
                references=targets_str,
                model_type=model_name,
                use_fast_tokenizer=True, 
                lang='en', 
                device=device,
                batch_size=batch_size,  # メモリ許容範囲で大きく設定
                rescale_with_baseline=False
                )['f1']
            bert_score = sum( bert_scores ) / len( bert_scores )

        inv_img = transforms_inv( imgs[0])
        plt.imshow( inv_img )
        plt.axis('off')
        plt.show()
        print( "hypo:", preds_str[0] )
        print( "refe:", targets_str[0] )
        print( "\n" )
        
        for pred, target in zip( preds_str, targets_str ):
            print( "hypo:", pred )
            print( "refe:", target )
  
        print( "these pics. cider :", avg_cider )
        print( "these pics. rouge:", avg_rouge )
        print( "these pics. clip_score:", clip_score )
        print( "these pics. bert_score:", bert_score )
        test_ciders.append(avg_cider)
        test_rouges.append(avg_rouge)
        test_clips.append(clip_score)
        test_berts.append(bert_score)
        n_iter += 1
        print(f'test number = {n_iter} average, cider = {torch.Tensor(test_ciders).mean().item()}, ' \
                f'rouge = {torch.Tensor(test_rouges).mean().item()}, clip_score = {torch.Tensor(test_clips).mean().item()}, ' \
                f'bert_score = {torch.Tensor(test_berts).mean().item()}' )
        print( "\n\n" )
        
            
        #if len(test_errors) > config.moving_avg:
        if len(test_ciders) > 100:
            test_ciders.popleft()
            test_rouges.popleft()
            test_clips.popleft()
            test_berts.popleft()
        pbar.set_postfix({
            #'loss': torch.Tensor(test_losses).mean().item(),
            'cider': torch.Tensor(test_ciders).mean().item(),
            'rougeL': torch.Tensor(test_rouges).mean().item(),
            'clip_score': torch.Tensor(test_clips).mean().item(),
            'bert_score': torch.Tensor(test_berts).mean().item()
        })                

# 表示
test_cider = np.mean( test_ciders )
test_rouge = np.mean( test_rouges )
test_clip = np.mean( test_clips )
test_bert = np.mean( test_berts )
print(f'test {n_iter} * {batch_size} average cider : {test_cider}')
print(f'test {n_iter} * {batch_size} average rougeL: {test_rouge}')
print(f'test {n_iter} * {batch_size} average clip_score: {test_clip}')
print(f'test {n_iter} * {batch_size} average bert_score: {test_bert}')